# Parent-car constraints for persistent part edits

Does an independently tracked whole car prevent a tagged door/window edit from landing on the other car? This notebook reproduces the fixed Day 5 ablation: raw SAM2 child masks versus those same masks intersected with independently tracked parent masks. Both edit arms protect every other original child mask, so parent clipping never releases an overlap.

Choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order. No training or parameter search occurs. The run uses three smoke clips and all twelve frozen test clips, each 64 frames at 384 × 384. It downloads the pinned source and official SAM2.1 Tiny checkpoint. The parent arm adds two whole-car frame-zero annotations and a separate tracking pass.

This repository notebook has **empty outputs** and has not been separately executed end to end on a clean GPU runtime. Its underlying experiment commands are being executed in the research Colab; measured evidence belongs in the saved run outputs. Installation and runtime availability may vary.

The benchmark is synthetic 2-D video. It does not establish real-video generalization, face consistency, automatic part discovery, or generative video editing. Test results never select parameters.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, shutil, subprocess, sys

REPO_URL = "https://github.com/skbdps/vjepa-poc.git"
REPO_REV = "de419ad916e57e04271c14fd830a8a52022b764e"
REPO = Path("/content/vjepa-poc-day5-parent")
RUN_ROOT = Path("/content/day5_parent_run")

if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", REPO_REV], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REPO_REV], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_REV
DAY4 = REPO / "experiments/day4"
DAY5 = REPO / "experiments/day5"
RUNNER = DAY5 / "run_parent_experiment.py"
FROZEN = DAY5 / "frozen_config.json"
for folder in (DAY4, DAY5):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Pinned experiment revision:", REPO_REV)
print("Run directory:", RUN_ROOT)

## Prepare the official frozen model

Keep Colab's matching PyTorch/torchvision installation. The official setup helper checks out the exact SAM2 source revision, disables the optional CUDA extension and downloads the official Tiny checkpoint. The runner independently verifies its source digest, policy, model revision and checkpoint hash before each stage. Do not regenerate the frozen configuration to make a mismatch disappear.

In [ ]:
from importlib.util import find_spec
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
missing = [package for module, package in {
    "numpy": "numpy>=1.24", "cv2": "opencv-python-headless==4.11.0.86", "PIL": "pillow"
}.items() if find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
import real_video
import run_parent_experiment as experiment

checkpoint = real_video.prepare_sam2(root="/content/sam2", checkpoint_dir="/content/checkpoints")
assert real_video.sha256_file(checkpoint) == experiment.CHECKPOINT_SHA256
frozen = experiment.validate_freeze(FROZEN)
assert frozen["model_revision"] == "2b90b9f5ceec907a1c18123530e92e794ad901a4"
shutil.copy2(FROZEN, RUN_ROOT / "frozen_config.json")
shutil.copy2(DAY5 / "PROTOCOL.md", RUN_ROOT / "PROTOCOL.md")
metadata = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "repo_url": REPO_URL, "repo_revision": REPO_REV,
    "source_digest": frozen["source_digest"],
    "checkpoint_sha256": experiment.CHECKPOINT_SHA256,
    "gpu": torch.cuda.get_device_name(), "torch": torch.__version__,
    "python": platform.python_version(), "precision": "float32",
    "prompt_frames": [0], "parameter_search": "none",
}
real_video.write_json(RUN_ROOT / "notebook_run_manifest.json", metadata)
print("Source digest:", frozen["source_digest"])
print("GPU:", metadata["gpu"])
print("Checkpoint SHA-256:", metadata["checkpoint_sha256"])

## CPU checks, then the three fixed smoke clips

The self-test uses small fabricated masks and no held-out scenes. The smoke stage checks execution and reporting; it must not be used to tune the policy. A fresh model is built from the verified checkpoint once per stage, then reused across clips; child and parent tracker states remain separate.

Each completed clip has checksum-verified caches. Rerunning a stage with the same source, frames, prompts and checkpoint reuses them. A partially inferred clip may rerun. Do not change the experiment after inspecting test outcomes.

In [ ]:
checks = experiment.self_test()
real_video.write_json(RUN_ROOT / "runner_self_test.json", checks)
print(json.dumps(checks, indent=2))

def run_stage(stage):
    command = [sys.executable, "-u", str(RUNNER), "--stage", stage,
               "--out", str(RUN_ROOT), "--checkpoint", checkpoint,
               "--frozen-path", str(FROZEN)]
    log_path = RUN_ROOT / f"{stage}.log"
    print("Running:", " ".join(command), flush=True)
    with log_path.open("a") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    report = json.loads((RUN_ROOT / stage / "results.json").read_text())
    assert report["frozen_config"] == frozen
    return report

smoke_report = run_stage("smoke")
assert len(smoke_report["clips"]) == 3
print("Smoke mechanics complete; no policy or threshold changes.")

## Execute the complete frozen test

Run all twelve clips even if early examples look unsuccessful. There are exactly **3,024 scored part-frames per arm and representation**, excluding frame zero. The primary outcome is absolute wrong-car effective paint pixels. Passing also requires at most 1 percentage point loss in visible mean IoU and pooled recall, separately for raw masks and effective edits.

Parent tracking costs an additional inference pass. Expect this cell to take substantially longer than the smoke cell; progress prints after each child/parent pass and completed clip.

In [ ]:
test_report = run_stage("test")
assert len(test_report["clips"]) == 12
for arm in experiment.ARMS:
    for representation in experiment.REPRESENTATIONS:
        assert test_report["arms"][arm][representation]["overall"]["scored_part_frames"] == 3024
metadata["completed_utc"] = datetime.now(timezone.utc).isoformat()
metadata["complete_test_clips"] = len(test_report["clips"])
real_video.write_json(RUN_ROOT / "notebook_run_manifest.json", metadata)
print("All twelve test clips complete. Fixed success assessment:")
print(json.dumps(test_report["success"], indent=2))

## Inspect measured results and the fixed crossing diagnostic

Dense raw masks, effective edit regions and whole-parent accuracy have distinct metrics. The secondary Day 4 patch score has a different two-frame denominator and does not gate editing. Visible slivers, complete absence, and paired whole-clip bootstrap intervals are reported separately.

The side-by-side video colors the four effective edit masks. **Magenta marks pixels placed on the other car's full visible ground-truth body.** This is a diagnostic mask visualization, not generated video. The displayed clip is the first crossing seed fixed by the protocol, not a selected best example.

In [ ]:
from IPython.display import display, Markdown, Video
for stage in ("smoke", "test"):
    display(Markdown((RUN_ROOT / stage / "RESULTS.md").read_text()))
    videos = sorted((RUN_ROOT / stage).glob("*_crossing_*_comparison.mp4"))
    assert len(videos) == 1
    display(Video(str(videos[0]), embed=True, width=768))

## Export the complete auditable run without frames or model weights

The ZIP includes child/parent prediction caches, patch predictions, manifests, per-frame CSVs, reports, diagnostic MP4/PNG files, logs, the original frozen policy, and exact source files required by its digest. Synthetic frames can be regenerated from seeds; weights can be downloaded from the official source. Neither is copied into the archive.

Download the archive before the temporary Colab runtime disappears. A SHA-256 inventory makes omissions and altered files detectable. Keep notebook outputs out of Git and preserve the measured run files separately.

In [ ]:
import zipfile
from google.colab import files

# Preserve exactly the source files covered by the freeze, retaining day4/day5 paths.
for relative, expected_hash in frozen["source_hashes"].items():
    source = REPO / "experiments" / relative
    assert real_video.sha256_file(source) == expected_hash
    target = RUN_ROOT / "source" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)

def include(path):
    relative = path.relative_to(RUN_ROOT)
    return (path.is_file() and "frames" not in relative.parts
            and path.suffix.lower() not in {".pt", ".pth", ".safetensors"}
            and path.name != "artifact_inventory.json")

paths = [path for path in sorted(RUN_ROOT.rglob("*")) if include(path)]
inventory = [{"path": str(path.relative_to(RUN_ROOT)), "bytes": path.stat().st_size,
              "sha256": real_video.sha256_file(path)} for path in paths]
real_video.write_json(RUN_ROOT / "artifact_inventory.json", inventory)
archive = Path("/content/VJEPA_Day5_Parent_Constraint_Run.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in paths + [RUN_ROOT / "artifact_inventory.json"]:
        bundle.write(path, arcname=str(path.relative_to(RUN_ROOT)))
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print("Archive:", archive, f"{archive.stat().st_size / 2**20:.2f} MiB")
print("Archive SHA-256:", real_video.sha256_file(archive))
files.download(str(archive))